# PCB Routing ACO

Ant colony optimization routes a single PCB net through an obstacle grid. Customize board dimensions, pin coordinates, obstacles, and ACO parameters. A route is reported if one is found.

Run the code cell to generate **`pcb-routing-aco.png`** in the repository folder.

In [ ]:
# Editable inputs: grid coordinates are (row, column)
ROWS,COLS=18,24
SOURCE,TARGET=(1,1),(16,22)
OBSTACLES=[(r,11) for r in range(3,15) if r not in (7,8,12)]+[(7,c) for c in range(4,9)]
ANTS,ITERATIONS,ALPHA,BETA,EVAP,SEED=35,100,1.0,3.0,.25,7
OUTPUT="pcb-routing-aco.png"
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(SEED)
blocked=set(map(tuple,OBSTACLES)); blocked.discard(SOURCE); blocked.discard(TARGET); pher=np.ones((ROWS,COLS)); moves=[(-1,0),(1,0),(0,-1),(0,1)]; best=None; hist=[]
def tour():
 p=SOURCE; path=[p]; seen={p}
 for _ in range(ROWS*COLS):
  if p==TARGET:return path
  opts=[(p[0]+dr,p[1]+dc) for dr,dc in moves]; opts=[q for q in opts if 0<=q[0]<ROWS and 0<=q[1]<COLS and q not in blocked and q not in seen]
  if not opts:return None
  w=np.array([pher[q]**ALPHA/(abs(q[0]-TARGET[0])+abs(q[1]-TARGET[1])+1e-6)**BETA for q in opts]); w/=w.sum(); p=opts[rng.choice(len(opts),p=w)]; seen.add(p); path.append(p)
 return None
for _ in range(ITERATIONS):
 paths=[p for p in (tour() for _ in range(ANTS)) if p]
 for p in paths:
  if best is None or len(p)<len(best): best=p
 pher*=1-EVAP
 for p in paths:
  for q in p: pher[q]+=1/(len(p)-1)
 hist.append(len(best)-1 if best else np.nan)
if best is None: raise RuntimeError("No path found; adjust obstacles or pins")
print("Route steps:",len(best)-1)
board=np.zeros((ROWS,COLS));
for q in blocked:board[q]=1
fig,(a,b)=plt.subplots(1,2,figsize=(12,5)); a.imshow(board,cmap="Greys"); a.plot([q[1] for q in best],[q[0] for q in best],"r.-"); a.scatter([SOURCE[1],TARGET[1]],[SOURCE[0],TARGET[0]],c=["green","blue"],s=90); a.set(title="PCB route",xlabel="Column",ylabel="Row"); b.plot(hist); b.set(title="Best route length",xlabel="Iteration",ylabel="Steps"); b.grid(alpha=.2); fig.tight_layout(); fig.savefig(OUTPUT,dpi=160); plt.show(); print("Saved",OUTPUT)